# Quick Start

This notebook gives a minimal end-to-end example: build a `ConvolsData` field from particle positions, smooth the fluctuation field, and estimate a simple two-point statistic from shell convolutions.


In [ ]:
from pyhermes.base.convols import Convols
from pyhermes.param.parambase import read_param
from pyhermes.io import WindowFunc
import matplotlib.pyplot as plt
import numpy as np

from pathlib import Path
import os
os.chdir(Path.cwd().resolve().parent)
print(f"Working directory: {Path.cwd()}")

In [ ]:
# Step 1: build the multiresolution coefficient field from particle positions
task_params = read_param(config_path="./configs/param_convols.yaml")
D = Convols(task_params).run(save_result=False)
D.threads = 8

# Step 2: normalize at the estimator boundary and construct delta = d - r
D_stat = D
rho = 1 / D_stat.V # uniform random field for a unit-total-weight estimator field
RR = rho ** 2 # <RR>, used to normalize the 2PCF estimator
deltaD = D_stat - rho # fluctuation field, i.e. the overdensity-like field

# Step 3: smooth the fluctuation field with a spherical window
win_params = {"type": "sphere", "len_args": {"R": 5}} # spherical top-hat window of radius 5
win_filter = WindowFunc(win_params, D.convols_info, threads=8)
deltaD_w = deltaD @ win_filter # smoothed fluctuation field

# Step 4: measure xi(r) by shell convolution + spatial averaging
# DD(r) = <n(x)n(x+r)>; n(x+r) = n_r(x) = n(x) @ W_shell(r)
# \xi = (d-r)(d-r)/RR
r_arr = np.linspace(0, 150, 26)
xi_arr = np.zeros_like(r_arr)
for i, r in enumerate(r_arr):
    pair_win_params = {"type": "shell", "len_args": {"R": r}}
    pair_win_shell = WindowFunc(pair_win_params, D.convols_info, threads=8)
    deltaDD_w = deltaD_w @ pair_win_shell * deltaD_w # shell-convolved field multiplied by the original field
    xi_arr[i] = deltaDD_w.as_array().mean() / RR # spatial mean gives xi(r) after normalization by RR

plt.plot(r_arr, xi_arr * r_arr**2)
plt.show()